<a href="https://colab.research.google.com/github/BosunL/SEA-VQA/blob/main/Swahili_Gemma_3_4B_SEA_CVQA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Visual Question Answering — Gemma 3 4B (Swahili)

---



Evaluating Gemma 3 4B on a custom Kenyan cultural dataset with 50 questions in Ateso across 14 images.

### Install Necessary Libraries

First, we need to install the `transformers` library to access pre-trained VQA models and `Pillow` for image processing.### 1. Install Necessary Libraries

In [ ]:
# Install transformers and other necessary libraries
!pip install -q transformers accelerate pillow matplotlib requests

### Import Libraries

Import the required modules from `transformers` and `PIL` (Pillow).

In [ ]:
#STEP 1: Install and Import Libraries

import random
import io
import requests
import torch
import matplotlib.pyplot as plt
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from huggingface_hub import login

print("Libraries imported successfully.")

###  Authenticate with Hugging Face

Accept the Gemma license at https://huggingface.co/google/gemma-3-4b-it then run this cell.

In [ ]:
# ============================================================
# STEP 2: Authenticate with Hugging Face
# ============================================================
# You need to accept the Gemma license at:
# https://huggingface.co/google/gemma-3-4b-it
# Then paste your HF token below or use the Colab secrets manager

login(token="INSERT TOKEN")


### Load Pre-trained VQA Model and Processor

We'll use the `Google/gemma-3-4b-it` model, which is a powerful VQA model.

In [ ]:
# ============================================================
# STEP 3: Load Model and Processor
# ============================================================

print("Loading Gemma 3 4B model and processor...")

MODEL_ID = "google/gemma-3-4b-it"

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print(f"Model loaded on: {model.device}\n")


### Load Custom Dataset (Swahili)

All 50 Ateso questions from the Kenyan culture dataset. Images are fetched from Google Drive by file ID and cached to avoid redundant downloads.


**PSA: Gemma doesn't handle Google Sheets well, so we had to manually write out the questions. This is why we ran multiple sanity checks on the model, making there be more steps in the process.**

In [ ]:
# ============================================================
# STEP 4: Load Custom Kenyan Culture Dataset (Swahili)
# ============================================================
# Images are loaded from Google Drive using the file ID.
# A cache dict avoids re-downloading the same image multiple times.

CUSTOM_DATA = [
    {
        "ID": "custom_001",
        "Category": "Image 1 Questions",
        "file_id": "14pBCQkb9v2VCsBEsE5NWlpMHrWx_FoN-",
        "swa_question": "Ni vikapu gani kati ya hivyo vilivyoonyeshwa, vinaitwaje katika nchi hii, na kusudi lao ni lipi?",
        "correct_swa": "Uteo. Imetengenezwa kwa matete au majani ya mitende. Kijadi hupatikana majumbani na ilitumika kutenganisha makapi na nafaka. Kwa kawaida hutumika kuhifadhi nafaka, sokoni, na majumbani.",
        "wrong_swa_o1": "Vikapu vya Mesob. Vipande vya nyasi au mitende hupakwa rangi angavu na hutumika kwa huduma ya pamoja ya kuandalia milo ya kitamaduni.",
        "wrong_swa_o2": "Vikapu vya Bolga. Vimetengenezwa kwa nyasi nene na kavu. Pia ni vya kudumu sana kwa sababu ya kusuka kwa ukali na hutumika kama mifuko ya sokoni au ufukweni, mapambo ya nyumbani, na kwa ajili ya kuhifadhi na kupanga.",
        "wrong_swa_o3": "Emotokaa. Kanueraitere nakapoloni itomate adakia inyamata luipwaka katoni luitijimete inyamene ayari namakegwelete.",
    },
    {
        "ID": "custom_002",
        "Category": "Image 1 Questions",
        "file_id": "14pBCQkb9v2VCsBEsE5NWlpMHrWx_FoN-",
        "swa_question": "Makopo ya njano ya jerry hutumika kwa ajili ya nini?",
        "correct_swa": "Uhifadhi wa mafuta ya kupikia, maji, na nafaka",
        "wrong_swa_o1": "Usafirishaji rahisi wa vikapu",
        "wrong_swa_o2": "Zinatumika kuhifadhi unga",
        "wrong_swa_o3": "Ongeza muda wa matumizi ya samaki mbichi na nyama mbichi",
    },
    {
        "ID": "custom_003",
        "Category": "Image 1 Questions",
        "file_id": "14pBCQkb9v2VCsBEsE5NWlpMHrWx_FoN-",
        "swa_question": "Ni aina gani ya usafiri ambayo ni ya kawaida kwa wachuuzi wadogo wa soko?",
        "correct_swa": "Baiskeli. Ni nafuu sana kwa sababu haitegemei mafuta. Inasafirisha mavuno madogo sokoni kwa urahisi, na hivyo kuruhusu wachuuzi kuuza bidhaa zao na kusafiri kwa urahisi wanaporudi.",
        "wrong_swa_o1": "Tuktuk. Ni nafuu na inaruhusu usafirishaji rahisi wa vitu vingi.",
        "wrong_swa_o2": "Gari. Kwa kuwa ndilo kubwa zaidi, wachuuzi wa soko mara nyingi husafirisha kiasi kikubwa cha chakula na viungo kutoka kwa mavuno ya hivi karibuni hadi mauzo sokoni.",
        "wrong_swa_o3": "Boda-boda. Pikipiki au teksi ya baiskeli yenye magurudumu mawili inaruhusu usafiri wa haraka zaidi kwa wateja wanaosubiri na inapatikana katika maeneo mengi ya vijijini na mijini.",
    },
    {
        "ID": "custom_004",
        "Category": "Image 1 Questions",
        "file_id": "14pBCQkb9v2VCsBEsE5NWlpMHrWx_FoN-",
        "swa_question": "Umuhimu wa gunia ni upi?",
        "correct_swa": "Gunia mara nyingi hutumika kuhifadhi mihogo na mahindi. Mifuko inaweza kutengenezwa kwa plastiki na safu ya nje iliyosokotwa vizuri.",
        "wrong_swa_o1": "Gunia hilo hutumika mara nyingi kuhifadhi wali, ambao hupikwa kama mlo wa kila siku na hutengenezwa kwa wingi.",
        "wrong_swa_o2": "Kwa kawaida gunia huhifadhi maharagwe makavu na huhifadhi virutubisho na ubora zaidi.",
        "wrong_swa_o3": "Gunia hutumika kukaushia mbegu zenye unyevu. Husaidia kuandaa mbegu za kupanda, na hivyo kusababisha mazao bora zaidi.",
    },
    {
        "ID": "custom_005",
        "Category": "Image 2 Questions",
        "file_id": "1x8AKBikOaDExx_LhB-vvuvLQZfuYBpeM",
        "swa_question": "Chombo cha bluu kilicho kwenye kona ya juu kulia kina vipande vingi vya chakula fulani vilivyokauka na kukatwakatwa. Ni chakula gani kinachotengenezwa kutokana na nyenzo hii, na ni mchakato gani uliotumika kufikia hatua hii?",
        "correct_swa": "Uji. Huwekwa sakafuni katika tabaka nyembamba juu ya kitambaa, huachwa kikauke juani, kisha huvunjwa vipande vidogo.",
        "wrong_swa_o1": "Mkate. Huchanganywa katika unga, huchovya katika mafuta ya kukaangia, kisha huachwa zipumzike hadi zipoe.",
        "wrong_swa_o2": "Chipsi. Hukatwa vipande vipande nyembamba, huokwa katika oveni kwa moto mkali hadi unyevunyevu wote utoke, kisha huchovya kwenye maji ili kupoa.",
        "wrong_swa_o3": "Wali wa kukaanga. Nafaka ambazo hazijapikwa huwekwa mfululizo, husagwa kwa kutumia rola, na kuchemshwa.",
    },
    {
        "ID": "custom_006",
        "Category": "Image 2 Questions",
        "file_id": "1x8AKBikOaDExx_LhB-vvuvLQZfuYBpeM",
        "swa_question": "Maandalizi na matumizi ya vyakula vingi vinavyopatikana masokoni yanajumuisha viungo na viungo kama vile bizari, manjano, giligilani, na tangawizi. Wasifu huu wa vyakula unaonyesha uhusiano gani wa kihistoria wa kibiashara?",
        "correct_swa": "Wafanyakazi wa reli na wafanyabiashara wa India wakiishi Afrika Mashariki.",
        "wrong_swa_o1": "Raia wa Uingereza wakiwafundisha Wakenya kuoka na kuchoma.",
        "wrong_swa_o2": "Wafanyabiashara wa Yoruba wakijiunga na masoko ya Kenya.",
        "wrong_swa_o3": "Walowezi wa Marekani wakileta mawazo ya kuhifadhi na kuchakata tena.",
    },
    {
        "ID": "custom_007",
        "Category": "Image 2 Questions",
        "file_id": "1x8AKBikOaDExx_LhB-vvuvLQZfuYBpeM",
        "swa_question": "Kwa kuzingatia makopo kwenye picha, yanaitwaje hapa?",
        "correct_swa": "Gorogoro",
        "wrong_swa_o1": "Kipimo",
        "wrong_swa_o2": "Uzito",
        "wrong_swa_o3": "Nafaka",
    },
    {
        "ID": "custom_008",
        "Category": "Image 2 Questions",
        "file_id": "1x8AKBikOaDExx_LhB-vvuvLQZfuYBpeM",
        "swa_question": "Ndoo na vyombo vya chuma hapo awali vilikuwa na vitu kama vile rangi, mbegu, na mafuta ya kupikia. Sasa vinatumika kwa nini? Hii inaonyesha nini?",
        "correct_swa": "Vipimo na uhifadhi. Huonyesha kuchakata na kutumia tena.",
        "wrong_swa_o1": "Uhifadhi. Huonyesha uhifadhi na uhifadhi wa hali ya juu wa majokofu.",
        "wrong_swa_o2": "Maonyesho na chapa ya soko. Inaonyesha biashara.",
        "wrong_swa_o3": "Kutengeneza chakula kinachoweza kubebeka. Huonyesha usafiri na uhamaji.",
    },
    {
        "ID": "custom_009",
        "Category": "Image 3 Questions",
        "file_id": "1LagZEieb5SzUKwQ2jLfgGOXhjchC-_B0",
        "swa_question": "Kwa nini muundo huu uko ndani ya maji?",
        "correct_swa": "Jengo hili lilikuwa mgahawa miaka michache iliyopita. Sasa, kilichobaki ni jengo la awali la chuma lililokuwa limesimama ndani ya maji. Hili liko katika Ziwa Victoria na kutokana na kupanda kwa kina cha bahari, mafuriko yamezidi kuathiri ardhi ya eneo hilo.",
        "wrong_swa_o1": "Jengo hili lilijengwa kwa sababu ndege waliendelea kuruka bara, wakipumzika kwenye nyumba za watu, na kusumbua kazi za kilimo.",
        "wrong_swa_o2": "Muundo huu hutumika kukamata samaki. Wavu ulio chini yake hunasa dagaa. Kwa bahati mbaya, muundo huo pia huvutia ndege.",
        "wrong_swa_o3": "Muundo huo hufanya kazi kama kitisho cha kunguru. Ndege hao huwaogopesha wadudu kutokana na kuingilia maeneo muhimu ya uvuvi kwa wavuvi.",
    },
    {
        "ID": "custom_010",
        "Category": "Image 3 Questions",
        "file_id": "1LagZEieb5SzUKwQ2jLfgGOXhjchC-_B0",
        "swa_question": "Ni aina gani ya ndege inayoonyeshwa na ndege mwenye kichwa cheusi mwenye mwili mweupe katikati ya picha?",
        "correct_swa": "Ibis takatifu za Kiafrika",
        "wrong_swa_o1": "Mchuzi Mkuu",
        "wrong_swa_o2": "Stork mwenye mdomo wa manjano",
        "wrong_swa_o3": "Komoranti",
    },
    {
        "ID": "custom_011",
        "Category": "Image 3 Questions",
        "file_id": "1LagZEieb5SzUKwQ2jLfgGOXhjchC-_B0",
        "swa_question": "Ni aina gani za ndege zinazoonekana kwenye picha?",
        "correct_swa": "Kwani mtakatifu wa Kiafrika, Mchuzi Mkuu, Korongo mwenye mdomo wa manjano, Cormorant",
        "wrong_swa_o1": "Korongo Mwenye Taji la Kijivu, Jacana wa Kiafrika, Shakwe Mdogo, Kibete",
        "wrong_swa_o2": "Kijidudu Kidogo, Mhunzi Plover, Mnyoofu Tern, Hadada Ibis",
        "wrong_swa_o3": "Bata Mzungu Mwenye Uso Mweupe, Coot Mwenye Vifundo Vikundu, Hamerkop, Korongo Mwenye Milia",
    },
    {
        "ID": "custom_012",
        "Category": "Image 4 Questions",
        "file_id": "1p6k5mfdOOWABJyA65VLM0TuoOPE7wF8I",
        "swa_question": "Ni matumizi gani ya kawaida ya boti zinazoonyeshwa mbele?",
        "correct_swa": "Uvuvi",
        "wrong_swa_o1": "Burudani",
        "wrong_swa_o2": "Usafiri wa baharini",
        "wrong_swa_o3": "Kusafirisha bidhaa",
    },
    {
        "ID": "custom_013",
        "Category": "Image 4 Questions",
        "file_id": "1p6k5mfdOOWABJyA65VLM0TuoOPE7wF8I",
        "swa_question": "Ni maji gani yanayoonyeshwa chinichini?",
        "correct_swa": "Ziwa Victoria",
        "wrong_swa_o1": "Bahari ya Hindi",
        "wrong_swa_o2": "Bahari Nyekundu",
        "wrong_swa_o3": "Maziwa Makuu",
    },
    {
        "ID": "custom_014",
        "Category": "Image 4 Questions",
        "file_id": "1p6k5mfdOOWABJyA65VLM0TuoOPE7wF8I",
        "swa_question": "Mwanamke aliye mbele kushoto amevaa kitambaa cha kitamaduni cha Kenya. Kinaitwaje?",
        "correct_swa": "Kitenge",
        "wrong_swa_o1": "Kente",
        "wrong_swa_o2": "Amauti",
        "wrong_swa_o3": "Agbada",
    },
    {
        "ID": "custom_015",
        "Category": "Image 4 Questions",
        "file_id": "1p6k5mfdOOWABJyA65VLM0TuoOPE7wF8I",
        "swa_question": "Kulingana na maandishi yanayopatikana kwenye boti hizo, picha hii imepigwa wapi Kenya?",
        "correct_swa": "Ufuo wa Dunga",
        "wrong_swa_o1": "Kiboko Point",
        "wrong_swa_o2": "Kit-Mikayi",
        "wrong_swa_o3": "Hifadhi ya Impala",
    },
    {
        "ID": "custom_016",
        "Category": "Image 5 Questions",
        "file_id": "1sd9LqsAAbxvlXVRG7c0TgyO4lTf0dLHT",
        "swa_question": "M-PESA imebadilisha vipi kimsingi ujumuishaji wa kifedha nchini Kenya?",
        "correct_swa": "Imebadilisha ujumuishaji wa kifedha wa Kenya kwani inaruhusu watu wasio na akaunti za benki kuweka, kutoa, kutuma, na kupokea pesa moja kwa moja kutoka kwa simu zao za mkononi kwa usalama.",
        "wrong_swa_o1": "Imefanya mazingira ya kifedha ya Kenya kuwa ya ubaguzi zaidi, kwani hutolewa kwa makundi maalum ya watu pekee.",
        "wrong_swa_o2": "Haijaleta tofauti kubwa au hakuna tofauti yoyote, kwani watu wengi hawaoni matumizi yake ya vitendo.",
        "wrong_swa_o3": "Imebadilisha kabisa ujumuishaji wa kifedha wa Kenya, kwani imesababisha mfumuko mkubwa wa bei kutokana na pesa nyingi zinazoweza kuingia na kutoka katika biashara.",
    },
    {
        "ID": "custom_017",
        "Category": "Image 5 Questions",
        "file_id": "1sd9LqsAAbxvlXVRG7c0TgyO4lTf0dLHT",
        "swa_question": "Vibanda hivi vya wauzaji wa ndani vinaitwaje?",
        "correct_swa": "Vibanda",
        "wrong_swa_o1": "Vibanda",
        "wrong_swa_o2": "Vibanda vya magazeti",
        "wrong_swa_o3": "Mapunguzo",
    },
    {
        "ID": "custom_018",
        "Category": "Image 5 Questions",
        "file_id": "1sd9LqsAAbxvlXVRG7c0TgyO4lTf0dLHT",
        "swa_question": "Je, kuna umuhimu gani wa Coca-Cola kuwekewa chapa kwenye muundo huo na nchini Kenya kwa ujumla?",
        "correct_swa": "Coca-Cola ndiyo kiwanda kikubwa zaidi cha chupa barani Afrika, kikiwa na kiwanda cha kuwekea chupa huko Kisumu. Coca-Cola huwapa wamiliki wa vibanda vifaa vya bure au vya ruzuku kama vile jokofu, miavuli, na maduka yaliyopakwa rangi. Kwa kubadilishana, wamiliki wa vibanda hukubali kuhifadhi bidhaa za Coca-Cola na kuonyesha nembo hiyo waziwazi.",
        "wrong_swa_o1": "Coca-Cola imenunua masoko yote ya ndani nchini Kenya ili kuunda ukiritimba juu ya soko la soda. Wanajitangaza wenyewe kwenye muundo huo ili kuwafanya raia watake kununua bidhaa zao mara nyingi zaidi.",
        "wrong_swa_o2": "Coca-Cola inafadhili biashara za ndani na ina motisha zinazowalipa watu kuvaa, kupaka rangi, na kuonyesha bidhaa zao hadharani. Hii imechochea sana masoko ya ndani lakini inatumika kama ukumbusho dhahiri kwamba makampuni yanazidi polepole maisha ya kila siku nchini Kenya.",
        "wrong_swa_o3": "Hakuna umuhimu wowote nyuma ya nembo hiyo kuchorwa; raia kote Kenya huonyesha upendo kwa chapa wanazozipenda kwa kuzichora kwenye nyumba zao, magari, na biashara zao.",
    },
    {
        "ID": "custom_019",
        "Category": "Image 5 Questions",
        "file_id": "1sd9LqsAAbxvlXVRG7c0TgyO4lTf0dLHT",
        "swa_question": "Ni hatua gani inayowezekana zaidi kufanyika katika jengo kama hili?",
        "correct_swa": "Kununua mazao madogo na chakula.",
        "wrong_swa_o1": "Kuuza nguo za zamani kwa pesa taslimu.",
        "wrong_swa_o2": "Wasilisha fomu na hati za kodi.",
        "wrong_swa_o3": "Kuweka usafiri hadi mahali.",
    },
    {
        "ID": "custom_020",
        "Category": "Image 6 Questions",
        "file_id": "1nATb92NkD0KfWqsiOsPCfeo9s-DXZB-m",
        "swa_question": "Ni miamba gani inayoonyeshwa kwenye picha hii, na umuhimu wake ni upi?",
        "correct_swa": "Mawe ya Kalsiamu kutoka Kodiaga. Mawe haya huliwa kijadi na wanawake wajawazito ili kuongeza kalsiamu, na kusaidia afya ya mifupa kwa mama na mtoto.",
        "wrong_swa_o1": "Mawe ya granite yanayotumika kwa ajili ya ujenzi. Kwa sababu granite ni nzito na hudumu, mawe yaliyoonyeshwa kwenye picha hutumika kwa ajili ya ujenzi wa kazi nzito na sakafu.",
        "wrong_swa_o2": "Chokaa inayopatikana kutoka sehemu zisizo na kina kirefu za Ziwa Victoria. Umbile lake laini hulifanya kuwa muhimu kwa ujenzi na kilimo. Inatumika kwa ajili ya kutibu maji machafu na kujaza chakula na dawa ya meno.",
        "wrong_swa_o3": "Mawe ya machimbo kutoka migodi iliyo karibu. Yameumbwa kwa sababu hali yake mbichi mara nyingi haitumiki. Hutumika mara kwa mara kwa ajili ya upinzani wa unyevu na ujenzi wa chini ya ardhi",
    },
    {
        "ID": "custom_021",
        "Category": "Image 6 Questions",
        "file_id": "1nATb92NkD0KfWqsiOsPCfeo9s-DXZB-m",
        "swa_question": "Mipira nyekundu iliyotengenezwa kwa plastiki ni nini na iliandaliwaje kwa ajili ya soko?",
        "correct_swa": "Karanga. Hutayarishwa, huloweshwa na maji ya moto, na kukorogwa na kupikwa kwenye sufuria yenye moto. Baada ya haya, ngozi hutiwa giza na kuchubuka. Hupozwa na kung'oka.",
        "wrong_swa_o1": "Maharage ya figo. Yamepangwa na kulowekwa. Yanachemshwa kwa dakika 10 ili kuondoa sumu. Hatimaye, moto hupunguzwa hadi kupikwa kidogo kwa chini ya saa moja hadi yawe laini.",
        "wrong_swa_o2": "Karanga. Hutayarishwa na kung'olewa. Maji kwenye sufuria hutiwa viungo yanapochemshwa. Maji yanapochemshwa, karanga hutiwa ndani na kupikwa kwa takriban saa tatu. Baada ya hapo, huloweshwa kwa saa chache ili chumvi iingie kwenye ganda. Huondolewa na kuuzwa.",
        "wrong_swa_o3": "Kokwa za makadamia. Huondolewa maganda kisha maganda hupasuka. Hukaushwa na kuokwa kwenye karatasi ya kuokea kwa dakika 10-12. Wakati wa mchakato huu, zinaweza kutiwa viungo. Huhifadhiwa na kuuzwa.",
    },
    {
        "ID": "custom_022",
        "Category": "Image 6 Questions",
        "file_id": "1nATb92NkD0KfWqsiOsPCfeo9s-DXZB-m",
        "swa_question": "Ni nini kilichomo kwenye ndoo nyeupe?",
        "correct_swa": "KSL Mints za Tropiki",
        "wrong_swa_o1": "Marumaru za kioo",
        "wrong_swa_o2": "Mawe ya bluu yanayotengenezwa kibiashara",
        "wrong_swa_o3": "Vito vya Tanzanite",
    },
    {
        "ID": "custom_023",
        "Category": "Image 7 Questions",
        "file_id": "1jLX5cxvjljTk0PtMOaXFG5m055KSCptC",
        "swa_question": "Ni mavazi gani yanayovaliwa katika picha hii?",
        "correct_swa": "Sketi za katani za Owali",
        "wrong_swa_o1": "Sketi za Hula kahiko",
        "wrong_swa_o2": "Sketi za Piupiu",
        "wrong_swa_o3": "Liku",
    },
    {
        "ID": "custom_024",
        "Category": "Image 7 Questions",
        "file_id": "1jLX5cxvjljTk0PtMOaXFG5m055KSCptC",
        "swa_question": "Mavazi haya yanawakilisha jamii gani?",
        "correct_swa": "Kiluo",
        "wrong_swa_o1": "Wamasai",
        "wrong_swa_o2": "Kisomali",
        "wrong_swa_o3": "Meru",
    },
    {
        "ID": "custom_025",
        "Category": "Image 7 Questions",
        "file_id": "1jLX5cxvjljTk0PtMOaXFG5m055KSCptC",
        "swa_question": "Mavazi haya huvaliwa lini na kwa ajili ya nini?",
        "correct_swa": "Na wanawake na hutumika kwa densi. Huvaliwa hasa wakati wa sherehe za ndoa, kusherehekea watoto wachanga, na unyago. Nyuzi hizo huru huruhusu harakati huru na sketi kutikisika na kutiririka kwa muziki.",
        "wrong_swa_o1": "Na watoto na hutumika kwa densi. Huvaliwa hasa kwenye sherehe za kuzaliwa kwa watoto na husherehekea uhuru wa ujana na sherehe ya maisha. Watoto hucheza kwenye miduara na hucheza nyatiti na abu",
        "wrong_swa_o2": "Na wanawake na wanaume na hutumika kwa ajili ya ibada. Huvaliwa hasa wakati wa nyakati takatifu kwa ajili ya dini na kukomaa. Jamii inaamini kwamba nyenzo zake zinawakilisha uumbaji wa Mungu, na hivyo ni mavazi matakatifu.",
        "wrong_swa_o3": "Na wanawake na hutumika kwa mitindo. Huvaliwa hasa kuonyesha rangi angavu zinazoonekana kama vile waridi angavu, zambarau, na bluu. Nyuzi zinazong'aa, zinazolainisha, na zinazokufa kutoka kwa mmea wa katani huifanya iwe nzuri sana na ufundi huu mzuri huifanya iwe vazi linalovaliwa kwa madhumuni ya mtindo na ya kuvutia.",
    },
    {
        "ID": "custom_026",
        "Category": "Image 8 Questions",
        "file_id": "1ZuWu-TWfzj_jE7c0Wgs8W3936X1Du-nQ",
        "swa_question": "Ni vitu gani vilivyowekwa kwenye mkanda mweusi au utepe?",
        "correct_swa": "Sifongo za asili za luffa",
        "wrong_swa_o1": "Mahindi meupe",
        "wrong_swa_o2": "Ndizi zilizokaushwa",
        "wrong_swa_o3": "Sifongo za baharini",
    },
    {
        "ID": "custom_027",
        "Category": "Image 8 Questions",
        "file_id": "1ZuWu-TWfzj_jE7c0Wgs8W3936X1Du-nQ",
        "swa_question": "Madhumuni ya vitu vilivyoshikiliwa kwenye utepe mweusi au bendi ni nini?",
        "correct_swa": "Visafishaji vya mwili. Vinaloweshwa kwenye maji na kisha vinafaa kwa ajili ya kusugua ngozi wakati wa kuoga. Pia ni vifaa vinavyooza na vya asili vya kusugua vyombo na kaunta.",
        "wrong_swa_o1": "Matumizi. Hutoa virutubisho bora kama vile potasiamu na vitamini B6. Kula hivyo huchangia afya bora ya utumbo na hivyo hufanya kazi kama chanzo bora cha chakula kwa wenyeji.",
        "wrong_swa_o2": "Kimiminika chao cha asili. Huhifadhi vimiminika vya asili vyenye madini mengi na wakati wa mavuno huchujwa kwa ajili ya vimiminika hivi. Huwekwa nje hadi vimiminika vyote kutoka mwilini vitakapokusanywa.",
        "wrong_swa_o3": "Ujenzi. Hupasuka na kuchanganywa na saruji ili kujaza mapengo kwenye kuta, sakafu, na paa. Ni nyenzo inayonyumbulika na imara inayoifanya iwe bora kwa ajili ya uboreshaji na marekebisho ya muundo wa kudumu kwa muda mrefu.",
    },
    {
        "ID": "custom_028",
        "Category": "Image 8 Questions",
        "file_id": "1ZuWu-TWfzj_jE7c0Wgs8W3936X1Du-nQ",
        "swa_question": "Mrundiko ni nini na upande wa kulia ni nini na vitu hivyo vinatumika kwa ajili ya nini?",
        "correct_swa": "Nyuzi mbichi za jute, ambazo hutumika kutengeneza vitambaa, mifuko, na mikeka. Pia zinaweza kubadilishwa kuwa uzi wa kufuma.",
        "wrong_swa_o1": "Majani ya ndizi yaliyokaushwa. Yana matumizi mengi, ikiwa ni pamoja na kusagwa kwa ajili ya chai. Pia hutumika kufunga viungo na vyakula vidogo kwa ajili ya kuhifadhi na kusokotwa kwenye ingata, ambayo hutumika wakati wa kubeba vitu vizito.",
        "wrong_swa_o2": "Mukeu, ambayo huvunwa ili kutengeneza kamba za asili zenye nguvu na za kudumu. Pia hutumika kusuka vikapu na mifuko ya kitamaduni ya kiondo",
        "wrong_swa_o3": "Mirundiko ya mikanda ya mpira iliyotengenezwa kienyeji. Inasaidia kwa kushikilia na kufunga bidhaa zinazouzwa kienyeji pamoja. Pia hutumika kwa ufundi, orthodontics, na usafiri.",
    },
    {
        "ID": "custom_029",
        "Category": "Image 9 Questions",
        "file_id": "1knmPao_ehrnkrqK_nOt5QRU1XAOAYaoL",
        "swa_question": "Ni aina gani ya samaki inayoonyeshwa kwenye picha?",
        "correct_swa": "Omena (Siprinidi ya Fedha)",
        "wrong_swa_o1": "Tilapia (Ngege)",
        "wrong_swa_o2": "Haplokromini (Fulu)",
        "wrong_swa_o3": "Ziwa Magadi Tilapia (Oreochromis alcalicus)",
    },
    {
        "ID": "custom_030",
        "Category": "Image 9 Questions",
        "file_id": "1knmPao_ehrnkrqK_nOt5QRU1XAOAYaoL",
        "swa_question": "Samaki huyu atatumika kwa nini baada ya kuuzwa?",
        "correct_swa": "Chakula cha binadamu na mifugo",
        "wrong_swa_o1": "Imesagwa ili kuuzwa katika vinywaji vya kienyeji",
        "wrong_swa_o2": "Chambo cha mchezo",
        "wrong_swa_o3": "Tiba pekee kwa magonjwa maalum",
    },
    {
        "ID": "custom_031",
        "Category": "Image 9 Questions",
        "file_id": "1knmPao_ehrnkrqK_nOt5QRU1XAOAYaoL",
        "swa_question": "Mbinu ya usindikaji hufanya nini?",
        "correct_swa": "Huongeza virutubisho, na kuvifanya kuwa chanzo kingi cha ulaji muhimu wa lishe",
        "wrong_swa_o1": "Huondoa vijidudu na uchafu wote, na kupunguza uwezekano wa kupata ugonjwa kutokana na ulaji.",
        "wrong_swa_o2": "Hutoa ladha asilia na kuifanya iwe ya kufurahisha zaidi kwa matumizi",
        "wrong_swa_o3": "Hulainisha na kurahisisha kupika kwenye vyombo",
    },
    {
        "ID": "custom_032",
        "Category": "Image 9 Questions",
        "file_id": "1knmPao_ehrnkrqK_nOt5QRU1XAOAYaoL",
        "swa_question": "Mwanamke aliyeonyeshwa kwenye picha anafanya nini?",
        "correct_swa": "Kuchagua samaki wadogo kwenye wavu wa matundu ili wakaushwe kwenye jua kwa ajili ya kuhifadhi",
        "wrong_swa_o1": "Kupanga samaki wadogo kwenye wavu wa matundu ili wawekwe kwenye sahani iliyopikwa kwa chakula cha jioni",
        "wrong_swa_o2": "Kuchagua samaki wadogo kwa kutumia wavu wa matundu ili kubaini ni samaki gani wenye ugonjwa au la",
        "wrong_swa_o3": "Kuondoa mifupa na ngozi ya samaki waliovunwa ili kuuzwa",
    },
    {
        "ID": "custom_033",
        "Category": "Image 10 Questions",
        "file_id": "1zah984l5GJlKEDbi8fF2qo9YSe4tspai",
        "swa_question": "Miundo hii imetengenezwa kwa nini?",
        "correct_swa": "Imetengenezwa kwa mbao, imefungwa kwa mchanganyiko wenye kinyesi cha ng'ombe. Paa pia zimetengenezwa kwa mbao, zimefungwa kwa ukali na nyasi kavu na matete, na kufungwa kwa kinyesi cha ng'ombe ili kuzuia mvua na mvua kuchakaa.",
        "wrong_swa_o1": "Imetengenezwa kwa mbao, imefungwa kwa saruji na matofali ya mbao. Paa pia zimetengenezwa kwa mbao, zimefungwa kwa nyasi na mianzi vizuri, na kufungwa kwa saruji ili kuifanya iwe imara.",
        "wrong_swa_o2": "Kuta zimeimarishwa kwa plasta ya jasi. Pembe zimepakwa mafuta ya mikaratusi ya limau ili kufukuza wadudu. Madirisha madogo na nafasi za kupoeza hewa. Paa pia zimepambwa kwa mbao na zimefungwa vizuri kwa nyasi kavu na matete.",
        "wrong_swa_o3": "Imetengenezwa kwa vijiti na kufungwa kwa udongo kwa ajili ya muundo imara. Msingi wake umejaa nyasi na matope ili iwe rahisi kulala. Nyasi huwekwa juu ya fremu ya paa iliyotengenezwa kwa mianzi.",
    },
    {
        "ID": "custom_034",
        "Category": "Image 10 Questions",
        "file_id": "1zah984l5GJlKEDbi8fF2qo9YSe4tspai",
        "swa_question": "Ni nani anayejenga majengo haya katika familia ya Waluo?",
        "correct_swa": "Wanaume na wanawake katika familia waligawanya majukumu",
        "wrong_swa_o1": "Wanawake katika familia",
        "wrong_swa_o2": "Mwana mkubwa",
        "wrong_swa_o3": "Wanaume katika familia (mume na wanawe)",
    },
    {
        "ID": "custom_035",
        "Category": "Image 10 Questions",
        "file_id": "1zah984l5GJlKEDbi8fF2qo9YSe4tspai",
        "swa_question": "Nani anaishi katika kibanda kikubwa zaidi?",
        "correct_swa": "Mke wa kwanza",
        "wrong_swa_o1": "Mume",
        "wrong_swa_o2": "Watoto",
        "wrong_swa_o3": "Wageni wanaotembelea",
    },
    {
        "ID": "custom_036",
        "Category": "Image 10 Questions",
        "file_id": "1zah984l5GJlKEDbi8fF2qo9YSe4tspai",
        "swa_question": "Huenda picha hii ilipigwa wapi?",
        "correct_swa": "Karibu na Kit Mikayi",
        "wrong_swa_o1": "Karibu na Maasai Mara",
        "wrong_swa_o2": "Karibu na Mlima Kenya",
        "wrong_swa_o3": "Karibu na Mkoa wa Pwani wa Kenya",
    },
    {
        "ID": "custom_037",
        "Category": "Image 10 Questions",
        "file_id": "1zah984l5GJlKEDbi8fF2qo9YSe4tspai",
        "swa_question": "Picha hii inaonyesha aina gani ya kibanda cha Kenya?",
        "correct_swa": "Od Mikayi",
        "wrong_swa_o1": "Manyatta / Enkaji",
        "wrong_swa_o2": "Mabawa ya Borana / Rendille",
        "wrong_swa_o3": "Vibanda vya Matumbawe vya Kiswahili",
    },
    {
        "ID": "custom_038",
        "Category": "Image 11 Questions",
        "file_id": "15A5HdRDwqnfbXayubjIrqsj5HR6dW0Kb",
        "swa_question": "Nguo za nani zinaonekana kwenye mstari?",
        "correct_swa": "Familia ya umri mbalimbali",
        "wrong_swa_o1": "Nguo za watoto tu",
        "wrong_swa_o2": "Nguo za wanawake pekee",
        "wrong_swa_o3": "Nguo za wanaume pekee",
    },
    {
        "ID": "custom_039",
        "Category": "Image 11 Questions",
        "file_id": "15A5HdRDwqnfbXayubjIrqsj5HR6dW0Kb",
        "swa_question": "Picha hii inasema nini kuhusu jinsi wenyeji wanavyoshughulikia nguo?",
        "correct_swa": "Wanaiosha kwa mikono na kuitundika kwenye kamba ya nguo",
        "wrong_swa_o1": "Wanatumia mashine za kufulia na kukaushia",
        "wrong_swa_o2": "Wanashona nguo zao wenyewe",
        "wrong_swa_o3": "Wanakusanya nguo na mara nyingi huziuza kwa ajili ya mapato",
    },
    {
        "ID": "custom_040",
        "Category": "Image 11 Questions",
        "file_id": "15A5HdRDwqnfbXayubjIrqsj5HR6dW0Kb",
        "swa_question": "Ni wanyama gani wanaoonekana chini ya kamba ya nguo na madhumuni yao ni nini?",
        "correct_swa": "Wao ni bata aina ya Muscovy na mara nyingi hufugwa na wakulima na wafugaji wa kuku wa mashambani. Ni ndege tulivu wanaotumika kwa nyama yao, mayai, na udhibiti wa wadudu waharibifu wa asili.",
        "wrong_swa_o1": "Wao ni Khaki Campbell na hutangatanga kutoka Ziwa Victoria lililo karibu na kuingia kwenye uwanja wa nyuma wa wanakijiji ili kula wadudu kwenye nyasi.",
        "wrong_swa_o2": "Ni kuku na mara nyingi hutumika kwa madhumuni ya kilimo. Hutoa mayai ya kutumia katika mapishi na kula kwa ajili ya virutubisho, na mara nyingi huuzwa au kuvunwa na wakulima kwa ajili ya nyama yao.",
        "wrong_swa_o3": "Ni aina ya bata vamizi wanaokula mimea na wadudu wanaowazunguka.",
    },
    {
        "ID": "custom_041",
        "Category": "Image 12 Questions",
        "file_id": "1UlOl57GPNS9cr8MoyA_xCKfbCvNVRPve",
        "swa_question": "Mwanamume huyu anafanya nini?",
        "correct_swa": "Kupiga chuma kwenye mashua ili kuijenga",
        "wrong_swa_o1": "Kutengeneza mashua ya mbao",
        "wrong_swa_o2": "Kujiandaa kuzindua mashua ndani ya maji.",
        "wrong_swa_o3": "Kujenga kreti ndefu ya mbao",
    },
    {
        "ID": "custom_042",
        "Category": "Image 12 Questions",
        "file_id": "1UlOl57GPNS9cr8MoyA_xCKfbCvNVRPve",
        "swa_question": "Kazi hii ina uwezekano mkubwa wa kuchukua jumla…",
        "correct_swa": "Wiki chache hadi miezi michache",
        "wrong_swa_o1": "Siku moja",
        "wrong_swa_o2": "Siku chache",
        "wrong_swa_o3": "Miaka michache",
    },
    {
        "ID": "custom_043",
        "Category": "Image 12 Questions",
        "file_id": "1UlOl57GPNS9cr8MoyA_xCKfbCvNVRPve",
        "swa_question": "Kazi ya mtu huyu ni",
        "correct_swa": "Mtengenezaji wa mashua",
        "wrong_swa_o1": "Mhandisi",
        "wrong_swa_o2": "Mvuvi",
        "wrong_swa_o3": "Mshonaji",
    },
    {
        "ID": "custom_044",
        "Category": "Image 13 Questions",
        "file_id": "1OpSdDSjVmev0-RXs18Y9_dnmEzJBzFMj",
        "swa_question": "Picha hii inaonyesha njia za kawaida za usafiri ni…",
        "correct_swa": "Boda-boda, tuktuks, magari",
        "wrong_swa_o1": "Pikipiki, mabasi madogo, reli",
        "wrong_swa_o2": "Baiskeli na pikipiki",
        "wrong_swa_o3": "Magari, pikipiki, vivuko",
    },
    {
        "ID": "custom_045",
        "Category": "Image 13 Questions",
        "file_id": "1OpSdDSjVmev0-RXs18Y9_dnmEzJBzFMj",
        "swa_question": "Teksi za pikipiki katika picha hii zinaitwaje?",
        "correct_swa": "Boda-boda",
        "wrong_swa_o1": "Baiskeli za magari",
        "wrong_swa_o2": "Okada",
        "wrong_swa_o3": "Mabaharia",
    },
    {
        "ID": "custom_046",
        "Category": "Image 13 Questions",
        "file_id": "1OpSdDSjVmev0-RXs18Y9_dnmEzJBzFMj",
        "swa_question": "Ni pikipiki gani inayoweza kuwa maarufu katika eneo hili?",
        "correct_swa": "Honda CG125",
        "wrong_swa_o1": "Ducati",
        "wrong_swa_o2": "BMW Motorrad",
        "wrong_swa_o3": "Honda NC750X",
    },
    {
        "ID": "custom_047",
        "Category": "Image 13 Questions",
        "file_id": "1OpSdDSjVmev0-RXs18Y9_dnmEzJBzFMj",
        "swa_question": "Madereva wa usafirishaji mara nyingi huendesha gari",
        "correct_swa": "Pikipiki au pikipiki",
        "wrong_swa_o1": "Magari",
        "wrong_swa_o2": "Malori",
        "wrong_swa_o3": "Tuktuk",
    },
    {
        "ID": "custom_048",
        "Category": "Image 14 Questions",
        "file_id": "15fdFEbGyGGQH5565RDtqjSPXCXWcIbyF",
        "swa_question": "Picha hii ilipigwa katika…",
        "correct_swa": "Kituo cha mafuta",
        "wrong_swa_o1": "Mkahawa",
        "wrong_swa_o2": "Urekebishaji wa kituo/magari ya fundi",
        "wrong_swa_o3": "Kituo cha kujaza maji",
    },
    {
        "ID": "custom_049",
        "Category": "Image 14 Questions",
        "file_id": "15fdFEbGyGGQH5565RDtqjSPXCXWcIbyF",
        "swa_question": "Ni nini kilichomo kwenye chupa kwenye rafu?",
        "correct_swa": "Sabuni / Kisafishaji cha Matumizi Mengi",
        "wrong_swa_o1": "Sabuni ya kufulia",
        "wrong_swa_o2": "Soda",
        "wrong_swa_o3": "Maji",
    },
    {
        "ID": "custom_050",
        "Category": "Image 14 Questions",
        "file_id": "15fdFEbGyGGQH5565RDtqjSPXCXWcIbyF",
        "swa_question": "Ni kampuni gani iliyoonyeshwa kwenye shati jekundu?",
        "correct_swa": "Shirika la Kitaifa la Mafuta la Kenya",
        "wrong_swa_o1": "Nishati ya Rubis Kenya",
        "wrong_swa_o2": "KFC",
        "wrong_swa_o3": "Safaricom",
    },
    {
        "ID": "custom_051",
        "Category": "Image 14 Questions",
        "file_id": "15fdFEbGyGGQH5565RDtqjSPXCXWcIbyF",
        "swa_question": "Kazi ya mtu huyu ni nini?",
        "correct_swa": "Kutoa mafuta na malipo ya usindikaji",
        "wrong_swa_o1": "Kuhudumia wateja wanaosubiri chakula",
        "wrong_swa_o2": "Kuendesha teksi na kuwa fundi",
        "wrong_swa_o3": "Kusafisha na kuwa mlinzi wa nyumba",
    },
]

def load_drive_image(file_id, _cache={}):
    """Download a Google Drive image by file ID, with caching."""
    if file_id in _cache:
        return _cache[file_id]
    url = f"https://drive.google.com/uc?export=download&id={file_id}"
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        img = Image.open(io.BytesIO(r.content)).convert("RGB")
        _cache[file_id] = img
        return img
    except Exception as e:
        print(f"  Warning: could not load image {file_id}: {e}")
        return Image.new("RGB", (224, 224), color="gray")

print(f"Custom dataset: {len(CUSTOM_DATA)} questions across 14 images.")


### Preview Dataset

In [ ]:
# ============================================================
# STEP 5: Preview Dataset — First 2 Samples
# ============================================================
print("--- Sample Preview: Custom Kenyan Dataset (Swahili) ---\n")

for i, sample in enumerate(CUSTOM_DATA[:2]):
    print(f"Sample {i+1}")
    print(f"  ID:         {sample['ID']}")
    print(f"  Category:   {sample['Category']}")
    print(f"  Question:   {sample['swa_question']}")
    print(f"  Correct:    {sample['correct_swa']}")
    print(f"  Wrong 1:    {sample['wrong_swa_o1']}")
    print(f"  Wrong 2:    {sample['wrong_swa_o2']}")
    print(f"  Wrong 3:    {sample['wrong_swa_o3']}")
    print()

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for i, sample in enumerate(CUSTOM_DATA[:2]):
    img = load_drive_image(sample['file_id'])
    axes[i].imshow(img)
    axes[i].set_title(f"Sample {i+1}\n{sample['Category'][:20]}", fontsize=8)
    axes[i].axis("off")
plt.suptitle("Custom Kenyan Dataset (Swahili) — First 2 Samples", fontsize=11)
plt.tight_layout()
plt.show()


### Helper Functions

In [ ]:
# ==========================================================
# STEP 6: Define Helper Functions
# ===========================================================

def build_mcqa_prompt(question, choices):
    """
    Format question + choices into a multiple choice prompt.
    Instructs the model to respond with only A, B, C, or D.
    """
    prompt = (
        "You are answering a visual multiple choice question. "
        "You must respond with ONLY a single letter: A, B, C, or D.\n\n"
        "Example:\n"
        "Question: What color is the sky?\n"
        "A. Red\nB. Blue\nC. Green\nD. Yellow\n"
        "Answer: B\n\n"
        "Now answer this question:\n"
        f"Question: {question}\n"
    )
    for label, choice in zip(["A", "B", "C", "D"], choices):
        prompt += f"{label}. {choice}\n"
    prompt += "Answer:"
    return prompt


def extract_predicted_label(predicted_text, choices):
    """
    Extract predicted label (A/B/C/D) from model output.
    Strategy 1: look for A/B/C/D letter in model output.
    Strategy 2: if no letter found, match answer text directly.
    """
    for label in ["A", "B", "C", "D"]:
        if label in predicted_text.upper():
            return label
    for label, choice in zip(["A", "B", "C", "D"], choices):
        if choice.lower() in predicted_text.lower():
            return label
    return None


### Gemma Inference Function

In [ ]:
# ==========================================================
# STEP 7: Helper Function to Run the Gemma Model
# ===========================================================

def run_gemma(image, prompt, processor, model):
    """
    Run Gemma 3 4B inference on an image + text prompt.
    """
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text": prompt}
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device, dtype=torch.bfloat16)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False
        )

    input_len = inputs["input_ids"].shape[-1]
    predicted_text = processor.decode(
        outputs[0][input_len:],
        skip_special_tokens=True
    ).strip()

    return predicted_text


### Evaluation Function

In [ ]:
# ============================================================
# STEP 8: Define Evaluation Function
# ============================================================

def evaluate_custom_dataset(data, processor, model):
    correct = 0
    predictions = []
    total = len(data)

    print(f"\nRunning evaluation on {total} samples...")
    print("-" * 40)

    for i, sample in enumerate(data):
        if (i + 1) % 10 == 0 or i == 0:
            print(f"  Processing sample {i+1}/{total}...")

        question    = sample["swa_question"]
        correct_ans = sample["correct_swa"]
        choices     = [
            correct_ans,
            sample["wrong_swa_o1"],
            sample["wrong_swa_o2"],
            sample["wrong_swa_o3"]
        ]
        random.shuffle(choices)
        correct_label = ["A", "B", "C", "D"][choices.index(correct_ans)]

        prompt = build_mcqa_prompt(question, choices)

        try:
            image           = load_drive_image(sample["file_id"])
            predicted_text  = run_gemma(image, prompt, processor, model)
            predicted_label = extract_predicted_label(predicted_text, choices)
            is_correct      = (predicted_label == correct_label)
            if is_correct:
                correct += 1
        except Exception as e:
            print(f"  Error on sample {i+1}: {e}")
            predicted_text  = "ERROR"
            predicted_label = None
            is_correct      = False

        predictions.append({
            "sample_id":       sample["ID"],
            "category":        sample["Category"],
            "question":        question,
            "choices":         {l: c for l, c in zip(["A","B","C","D"], choices)},
            "correct_answer":  correct_ans,
            "correct_label":   correct_label,
            "predicted_text":  predicted_text,
            "predicted_label": predicted_label,
            "is_correct":      is_correct
        })

    accuracy = (correct / total) * 100 if total > 0 else 0
    return accuracy, predictions


### Sanity Check (5 Samples)

In [ ]:
# ============================================================
# STEP 9: Sanity Check on First 5 Samples
# ============================================================
print("\nRunning sanity check on 5 samples...")

_, sanity_predictions = evaluate_custom_dataset(
    CUSTOM_DATA[:5], processor, model
)

print("\nSanity check results:")
for i, pred in enumerate(sanity_predictions, 1):
    status = "✓" if pred["is_correct"] else "✗"
    print(f"  [{status}] Q{i}: {pred['question']}")
    print(f"        Predicted: {pred['predicted_label']} "
          f"| Correct: {pred['correct_label']} "
          f"| Model output: {pred['predicted_text']}")

print("\nIf results look reasonable, proceed to full evaluation...\n")


### Full Evaluation (50 Questions)

In [ ]:
# ============================================================
# STEP 10: Run Full Evaluation (50 questions)
# ============================================================
accuracy, all_predictions = evaluate_custom_dataset(
    CUSTOM_DATA, processor, model
)


### Results

In [ ]:
# ============================================================
# STEP 11: Print Final Results
# ============================================================
print("\n" + "="*60)
print(f"  Final Results — Gemma 3 4B | Kenyan Culture Dataset (Swahili)")
print("="*60)

category_scores = {}
for pred in all_predictions:
    cat = pred["category"]
    if cat not in category_scores:
        category_scores[cat] = {"correct": 0, "total": 0}
    category_scores[cat]["total"] += 1
    if pred["is_correct"]:
        category_scores[cat]["correct"] += 1

print("\nPer-image accuracy:")
for cat, scores in sorted(category_scores.items()):
    cat_acc = (scores["correct"] / scores["total"]) * 100
    print(f"  {cat:<45} {cat_acc:.1f}%  ({scores['correct']}/{scores['total']})")

print(f"\nOverall Accuracy:        {accuracy:.2f}%")
print(f"Correct:                 {sum(p['is_correct'] for p in all_predictions)}/{len(all_predictions)}")
print(f"Random chance baseline:  25.00% (4 choices)")
print("="*60)


### Visualise Sample Results

In [ ]:
# ============================================================
# STEP 12: Visualise Results
# ============================================================

import math

# Create a dictionary to group predictions by image file_id
image_performance_map = {}
for i, sample in enumerate(CUSTOM_DATA):
    file_id = sample["file_id"]
    if file_id not in image_performance_map:
        image_performance_map[file_id] = {"image": None, "predictions": []}
    # Load image only once per file_id
    if image_performance_map[file_id]["image"] is None:
        image_performance_map[file_id]["image"] = load_drive_image(file_id)

    # Associate prediction with the correct image based on sample ID
    # Note: all_predictions is 1:1 with CUSTOM_DATA, so we can use index 'i'
    image_performance_map[file_id]["predictions"].append(all_predictions[i])

# Get a sorted list of unique file_ids for consistent plotting order
unique_file_ids = sorted(image_performance_map.keys())
num_unique_images = len(unique_file_ids)

# Determine grid dimensions (e.g., 4 columns)
ncols = 4
nrows = math.ceil(num_unique_images / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
# Flatten the axes array for easier iteration if it's a multi-row grid
axes = axes.flatten() if nrows > 1 or ncols > 1 else [axes] # Handle single subplot case

for i, file_id in enumerate(unique_file_ids):
    img_data = image_performance_map[file_id]
    img = img_data["image"]
    predictions_for_image = img_data["predictions"]

    correct_count = sum(p['is_correct'] for p in predictions_for_image)
    total_questions = len(predictions_for_image)
    image_accuracy = (correct_count / total_questions) * 100 if total_questions > 0 else 0

    # Use the category of the first question associated with this image as a general image label
    image_category = predictions_for_image[0]["category"]

    axes[i].imshow(img)
    color = "green" if image_accuracy == 100 else ("red" if image_accuracy == 0 else "orange")
    axes[i].set_title(
        f"{image_category}\nAcc: {image_accuracy:.1f}% ({correct_count}/{total_questions})",
        fontsize=8, color=color
    )
    axes[i].axis("off")

# Hide any unused subplots
for j in range(num_unique_images, len(axes)):
    if j < len(axes): # Ensure index is valid
        fig.delaxes(axes[j])

plt.suptitle(
    f"Gemma 3 4B — Kenyan Culture Dataset (Ateso) | Overall Accuracy: {accuracy:.1f}% | Random baseline: 25%",
    fontsize=12
)
plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to make space for suptitle
plt.show()